# Drift-aware fraud MLOps: replication on BAF (Bank Account Fraud, NeurIPS 2022)

Runs the thesis experiments (the same scripts and settings as for IEEE-CIS and Sparkov) on the
**BAF** dataset: baseline decay → label-delay sweep → walk-forward tuning → 5 seeds →
full framework replay (60-day and expanding training window).

**Before running**
1. **Add Input** → search *bank account fraud dataset suite neurips 2022* (by `sgpjesus`) → add it.
2. **Settings** → Accelerator **None** (CPU is enough), **Internet on** (needs a phone-verified account).
3. If the GitHub repo is private: either upload the code folder as a private Kaggle dataset and add
   it as an input (it is found automatically), or add a secret `GITHUB_TOKEN` (*Add-ons → Secrets*).
4. **Save Version → Save & Run All (Commit)**. It runs in the background; you can close the tab.
   Results: the version's **Output** tab → `baf_outputs.zip`.

**If a run hits Kaggle's 12-hour limit:** set `PART = "2"` below, add *this notebook's previous
version output* as an input (**Add Input → Your Work**), and commit again. Finished steps
and finished label delays/seeds are restored and skipped.

In [ ]:
# ---- settings ----
PART = "all"        # "all", "1" (prepare -> walk-forward), or "2" (seeds -> replays)
RUN_TESTS = True    # quick test suite first (about 2 min) to catch package problems early
REPO_URL = "https://github.com/nishan469/fraud-mlops-thesis.git"
VARIANT = "Base.csv"   # or e.g. "Variant II.csv" for one of BAF's biased variants

In [ ]:
import glob, os, shutil, subprocess, sys
WORK = "/kaggle/working"
CODE = f"{WORK}/fraud-mlops-thesis"

# 1) code: an attached Kaggle dataset with the code, else git clone (token from secrets if any)
if not os.path.exists(f"{CODE}/run_replication.py"):
    attached = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/run_replication.py", recursive=True)
                if "/outputs/" not in p]
    if attached:
        shutil.copytree(attached[0], CODE, ignore=shutil.ignore_patterns("outputs", "data", "mlruns", "mlflow.db"))
        print("Code copied from", attached[0])
    else:
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{token}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "--depth", "1", url, CODE], check=True)
        print("Code cloned from GitHub")
os.chdir(CODE)
print(sorted(os.listdir(".")))

In [ ]:
# 2) packages: install only what is missing or too old (Kaggle preinstalls most of them)
!pip install -q "lightgbm>=4.7" "mlflow>=3.16" pyarrow fastapi httpx pytest
import pandas, sklearn, lightgbm, mlflow, numpy
print("python", sys.version.split()[0], "| pandas", pandas.__version__, "| sklearn", sklearn.__version__,
      "| lightgbm", lightgbm.__version__, "| mlflow", mlflow.__version__, "| numpy", numpy.__version__)
print("CPUs:", os.cpu_count())

In [ ]:
# 3) quick test suite on synthetic data
if RUN_TESTS:
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-W", "ignore", "tests"],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    assert r.returncode == 0, "tests failed: fix packages before the long run"

In [ ]:
# 4) find the BAF files and restore finished work from a previous version, if attached
hits = [p for p in glob.glob(f"/kaggle/input/**/{VARIANT}", recursive=True)]
assert hits, f"{VARIANT} not found: add the BAF dataset as an input (see the top of the notebook)"
BAF_DIR = os.path.dirname(hits[0])
print("BAF data:", BAF_DIR)

prev = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/outputs/baf/logs", recursive=True)]
if prev:
    shutil.copytree(prev[0], "outputs/baf", dirs_exist_ok=True)
    print("Restored previous outputs from", prev[0])

In [ ]:
# 5) run. Each step logs to outputs/baf/logs/<step>.log; progress lines appear here.
PARTS = {"all": (None, None), "1": ("prepare", "walkforward"), "2": ("seeds", None)}
start, stop = PARTS[PART]
env = {**os.environ, "PYTHONUNBUFFERED": "1", "MLFLOW_DISABLE_AGENT_HINT": "1"}

def run(*args):
    p = subprocess.Popen([sys.executable, "run_replication.py", "--dataset", "baf", "--in_dir", BAF_DIR, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        log = sorted(glob.glob("outputs/baf/logs/*.log"), key=os.path.getmtime)[-1]
        print("".join(open(log, encoding="utf-8").readlines()[-40:]))
        raise RuntimeError(f"step failed, see {log}")

if start and start != "prepare" and not os.path.exists("data/baf/transactions.parquet"):
    run("--to", "prepare")            # the prepared table is not saved between sessions; rebuild it (1 min)
args = (["--from", start] if start else []) + (["--to", stop] if stop else [])
run(*args)

In [ ]:
# 6) results
import json
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200, "display.max_columns", 20)
O = "outputs/baf"
def show(title, path, **kw):
    if os.path.exists(path):
        print(); print(f"=== {title} ==="); display(pd.read_csv(path, **kw) if path.endswith(".csv") else None)

if os.path.exists(f"{O}/baseline/baseline_metrics.json"):
    print("=== Static model, test period ===")
    print(json.dumps(json.load(open(f"{O}/baseline/baseline_metrics.json"))["test_overall"], indent=1))
for p in ["baseline/degradation_by_time.png", "sweep/sweep_label_delay.png",
          "walkforward/wf_walkforward.png", "seeds/seed_robustness.png"]:
    if os.path.exists(f"{O}/{p}"):
        print(); print(p); display(Image(f"{O}/{p}", width=900))
show("Walk-forward: tuned trigger vs schedule vs static", f"{O}/walkforward/wf_summary.csv")
show("Seeds: paired differences", f"{O}/seeds/seed_diffs.csv")
for name in ["replay", "replay_expanding"]:
    p = f"{O}/{name}/summary.json"
    if os.path.exists(p):
        print(); print(f"=== Framework {name} ==="); print(json.dumps(json.load(open(p)), indent=1))

In [ ]:
# 7) one zip to download (Output tab -> baf_outputs.zip); big prediction caches left out
import zipfile
with zipfile.ZipFile(f"{WORK}/baf_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk("outputs/baf"):
        for f in files:
            if not f.endswith(".npz"):
                z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), "outputs"))
if os.path.exists("mlflow.db"):
    shutil.copy("mlflow.db", f"{WORK}/baf_mlflow.db")
print(f"{os.path.getsize(f'{WORK}/baf_outputs.zip') / 1e6:.0f} MB -> {WORK}/baf_outputs.zip")